# ♻️ Entrenamiento YOLO11 — TACO (multi-objeto) · v2 a prueba de cortes

**Iteración 2** del proyecto, con el dataset **TACO** (basura en escenas reales,
~2.8 objetos/imagen) para detectar **varios residuos por imagen**.

🛡️ **Novedad v2:** el entrenamiento se guarda **directamente en tu Google Drive**
y es **reanudable**. Si Colab corta por límite de GPU, NO pierdes el progreso:
solo reanudas desde el último checkpoint.

- **18 clases** · ~6,000 imágenes · CC BY 4.0

---
### ⚙️ ANTES DE EMPEZAR
1. **Entorno de ejecución → Cambiar tipo → GPU (T4)**.
2. Sube `TACO.v3-yolov5.yolo26.zip` a la **raíz de tu Google Drive**.

## 1. Verificar la GPU

In [ ]:
!nvidia-smi

## 2. Instalar Ultralytics

In [ ]:
!pip install ultralytics -q
import ultralytics
ultralytics.checks()

## 3. Montar Drive y preparar el dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile
from pathlib import Path

ZIP_EN_DRIVE = "/content/drive/MyDrive/TACO.v3-yolov5.yolo26.zip"
DESTINO = "/content/dataset_taco"

assert os.path.exists(ZIP_EN_DRIVE), f"No encuentro el ZIP en {ZIP_EN_DRIVE}."

os.makedirs(DESTINO, exist_ok=True)
with zipfile.ZipFile(ZIP_EN_DRIVE, "r") as z:
    z.extractall(DESTINO)

for split in ["train", "valid", "test"]:
    carpeta = Path(DESTINO) / split / "images"
    n = len(list(carpeta.glob("*.jpg"))) if carpeta.exists() else 0
    print(f"{split:6}: {n} imágenes")

### Ajustar `data.yaml` (lee las 18 clases automáticamente)

In [ ]:
import yaml
with open(f"{DESTINO}/data.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["train"] = f"{DESTINO}/train/images"
cfg["val"]   = f"{DESTINO}/valid/images"
cfg["test"]  = f"{DESTINO}/test/images"
cfg.pop("roboflow", None)
YAML_PATH = f"{DESTINO}/data.yaml"
with open(YAML_PATH, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
print("Nº de clases:", cfg["nc"]); print("Clases:", cfg["names"])

## 4. Entrenar YOLO11n (guardando en Drive)

🛡️ Clave: `project` apunta a tu **Drive**, así cada época se guarda ahí y NADA se
pierde si Colab corta. `save_period=10` guarda un checkpoint extra cada 10 épocas.
En T4 tarda ~1.5 h. Si se corta, salta a la celda **"Reanudar"** de abajo.

In [ ]:
from ultralytics import YOLO

PROJECT_DIR = "/content/drive/MyDrive/taco_runs"   # <- se guarda en tu Drive
RUN_DIR = f"{PROJECT_DIR}/residuos_taco"

model = YOLO("yolo11n.pt")
results = model.train(
    data=YAML_PATH,
    epochs=60,
    imgsz=640,
    batch=16,
    patience=20,
    project=PROJECT_DIR,
    name="residuos_taco",
    save_period=10,        # checkpoint cada 10 épocas
    plots=True,
)

## ▶️ ¿Se cortó por límite de GPU? Reanuda aquí

Cuando vuelvas a tener GPU (otra cuenta, Kaggle o tras ~12-24 h), ejecuta las
celdas 1-3 (instalar + montar Drive + preparar dataset) y luego **esta celda**.
Reanuda desde el último checkpoint guardado en tu Drive, **sin empezar de cero**.

In [ ]:
from ultralytics import YOLO

LAST = "/content/drive/MyDrive/taco_runs/residuos_taco/weights/last.pt"
model = YOLO(LAST)
results = model.train(resume=True)

## 5. Evaluar el modelo

In [ ]:
metrics = model.val()
print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precisión    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")

In [ ]:
import os
from IPython.display import Image as ColabImage, display
RUN_DIR = "/content/drive/MyDrive/taco_runs/residuos_taco"
for nombre in ["results.png", "confusion_matrix.png"]:
    ruta = os.path.join(RUN_DIR, nombre)
    if os.path.exists(ruta):
        print("==>", nombre); display(ColabImage(filename=ruta, width=820))

## 6. 🎯 Demo MULTI-OBJETO

In [ ]:
import glob, random
import matplotlib.pyplot as plt

TEST_DIR = "/content/dataset_taco/test/images"
imagenes = sorted(glob.glob(TEST_DIR + "/*.jpg"))
random.seed(7)
muestra = random.sample(imagenes, min(9, len(imagenes)))

fig, axes = plt.subplots(3, 3, figsize=(16, 16))
for ax, img_path in zip(axes.flat, muestra):
    r = model.predict(img_path, conf=0.25, verbose=False)[0]
    ax.imshow(r.plot()[..., ::-1])
    ax.set_title(f"{len(r.boxes)} objetos detectados", fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/taco_runs/demo_multiobjeto.png", dpi=110)
plt.show()
print("Guardado en tu Drive: taco_runs/demo_multiobjeto.png")

## 7. Descargar el modelo
Tu modelo ya está en Drive (`taco_runs/residuos_taco/weights/best.pt`). Esta
celda lo descarga también a tu PC.

In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/taco_runs/residuos_taco/weights/best.pt")